<a href="https://colab.research.google.com/github/venkatasai-eng/MLA0305-REINFORCEMENT-LEARNING-/blob/main/EXP_NO_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

states = 5
actions = 2
episodes = 100
gamma = 0.9

def step(s, a):
    ns = min(s + 1, 4) if a == 1 else max(s - 1, 0)
    r = 10 if ns == 4 else -1
    return ns, r, ns == 4

def encode(s):
    x = np.zeros(states)
    x[s] = 1
    return x

def create_model():
    inp = layers.Input(shape=(states,))
    x = layers.Dense(16, activation="relu")(inp)
    actor = layers.Dense(actions, activation="softmax")(x)
    critic = layers.Dense(1)(x)
    return tf.keras.Model(inp, [actor, critic])

def train(model):
    opt = tf.keras.optimizers.Adam(0.001)
    rewards = []

    for ep in range(episodes):
        s = 0
        total = 0

        while True:
            x = encode(s).reshape(1, -1)

            with tf.GradientTape() as tape:
                p, v = model(x)
                a = np.random.choice(actions, p=p[0].numpy())

                ns, r, done = step(s, a)
                _, nv = model(encode(ns).reshape(1, -1))

                target = r if done else r + gamma * nv[0, 0]
                adv = target - v[0, 0]

                loss = (
                    -tf.math.log(p[0, a] + 1e-8) * tf.stop_gradient(adv)
                    + 0.5 * tf.square(adv)
                )

            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))

            total += r
            s = ns

            if done:
                break

        rewards.append(total)

    return rewards

a2c = create_model()
a3c = create_model()

a2c_rewards = train(a2c)
a3c_rewards = train(a3c)

print("A2C Average Reward:",
      round(np.mean(a2c_rewards[-10:]), 2))

print("A3C Average Reward:",
      round(np.mean(a3c_rewards[-10:]), 2))

print("\nA2C Training Completed")
print("A3C Training Completed")

A2C Average Reward: 6.0
A3C Average Reward: 6.4

A2C Training Completed
A3C Training Completed
